In [1]:
import datasets
dataset=datasets.load_dataset('/Users/chenyao/Documents/DeepLiterature/data')


/Users/chenyao/opt/anaconda3/envs/sone/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset

DatasetDict({
    test: Dataset({
        features: ['question', 'source', 'true_answer', 'true_reasoning'],
        num_rows: 132
    })
})

In [3]:
sys_prompt="""\
# Instructions
Your task is to act as a rigorous grading expert. 
Based on the standard answer, determine whether the user's response is correct and explain why:
First, assess if the user’s answer is accurate, i.e., if it covers the core points of the standard answer.
If there are differences, point out exactly what is missing or incorrect.
Finally, give an overall judgment: Correct:1, or Incorrect:0.

# Type of Correct Answer
Number/Word/Sentence (as specified by the question)  

#Key Extraction Rules:  
   • Ignore explanations, examples, or descriptive language in the user's answer.  
   • Focus on the core content of the same type as the correct answer (e.g., if the correct answer is a number, extract only the numerical result from the user's answer).
   
Here is the content to evaluate:

Question: {question}

Standard Answer: {answer}

User Answer: {user_Answer}

# OUTPUT
{{"cot":<chain of thought>, "answer": <your judgment>}}\
"""

In [4]:
# 安装 OpenAI SDK：pip3 install openai
from openai import OpenAI
import os
import json
import concurrent.futures
from tqdm import tqdm
from openai import OpenAI
import time

# 创建 API 客户端
YOUR_API_KEY = "30a70266-37d5-4210-b8a2-34d5fb629230"
BASE_URL = "https://ark.cn-beijing.volces.com/api/v3"
# MODEL = "ep-20250321143411-fvc95"
MODEL = "ep-20250318205920-bz4vk"
### DeepSeek-V3
# ep-20250321143411-fvc95
client = OpenAI(api_key=YOUR_API_KEY, base_url=BASE_URL)

def get_openai_response(prompt, temperature=0.8, max_tokens=8000, retries=5):
    model = MODEL
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=prompt,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            content = response.choices[0].message.content.strip()
            return content

        except Exception as e:
            if attempt < retries - 1:
                pass
            else:
                print(f"Failed after {retries} attempts: {e}")
                return None

def get_openai_reasoning_response(prompt, temperature=0.8, max_tokens=8000, retries=5):
    model = MODEL
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=prompt,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            reasoning_content = response.choices[0].message.reasoning_content.strip()
            content = response.choices[0].message.content.strip()
            return reasoning_content, content

        except Exception as e:
            if attempt < retries - 1:
                pass
            else:
                print(f"Failed after {retries} attempts: {e}")
                return None



# 并行处理函数
def run_parallel_requests(
    prompts,
    get_openai_response=get_openai_reasoning_response,
    n=1,
    temperature=0.8,
    max_tokens=8192,
    max_workers=132,
):
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 创建请求任务并使用tqdm显示进度条
        # 创建请求任务并使用tqdm显示进度条，同时传递索引来保持顺序
        futures = {
            executor.submit(get_openai_response, prompt, temperature, max_tokens): index
            for index, prompt in enumerate(prompts)
        }
        results = [None] * len(futures)
        # 等待所有任务完成并按照提交顺序收集结果
        for future in tqdm(
            concurrent.futures.as_completed(futures), total=len(futures)
        ):
            try:
                result = future.result()
                index = futures[future]  # 获取任务对应的索引
                results[index] = result  # 将结果存储到正确的位置
            except Exception as e:
                index = futures[future]
                results[index] = None  # 如果出错，则存储None
    return results


In [5]:
with open("/Users/chenyao/Documents/DeepLiterature/src/logs/smol_test.jsonl", "rb") as f:
    data = f.readlines()
cnt=0
user_answer=[]

for i,line in enumerate(data):
    try:
        ans=list(json.loads(data[i].decode("utf-8"))['now_messages_ls'][-1].values())[0][-1]['content']
        user_answer.append(ans)
    except:
        cnt+=1
        user_answer.append('')
        
print(cnt)

24


In [6]:
dataset['test'] = dataset['test'].add_column("user_answer",user_answer)

In [7]:
def gen_prompts(example):
    # nonlocal sys_prompt
    prompt=sys_prompt.format(question=example["question"],answer=example["true_answer"],user_Answer=example["user_answer"])
    messages = [
            {"role": "user", "content": prompt},
        ]
    return {"prompt":messages}

In [8]:
dataset['test'] = dataset['test'].map(gen_prompts)

In [9]:
tmp_res=get_openai_reasoning_response(gen_prompts(dataset['test'][0])['prompt'])

In [10]:
json.loads(tmp_res[1])['answer']


0

In [11]:
res=run_parallel_requests(dataset['test']['prompt'],get_openai_response=get_openai_response)


100%|██████████| 132/132 [05:40<00:00,  2.58s/it]


In [ ]:
# res=[(res[i][0],res[i][1],dataset['test']['user_answer'][i],dataset['test']['source'][i]) for i in range(len(res))]
res=[(None,res[i],dataset['test']['user_answer'][i],dataset['test']['source'][i],dataset['test']['question'][i]) for i in range(len(res))]

In [38]:
new_res=[(res[i][0],res[i][1],dataset['test']['user_answer'][i],dataset['test']['source'][i],dataset['test']['true_answer'][i],dataset['test']['question'][i]) for i in range(len(res))]

In [39]:
new_res[0]

(None,
 '{"cot":"The user\'s answer of 17000 hours results from correctly rounding 17059.87 to the nearest 1000 hours. However, the question explicitly asks for the answer in \'thousand hours\' (i.e., the number of thousands), which requires dividing the total hours by 1000 after rounding. The standard answer is 17 (thousand hours), whereas the user provided 17000 (hours), missing the required unit conversion. Thus, the answer is incorrect.","answer":"0"}',
 '\n\n【Answer】The rounded result to the nearest 1000 hours (without commas) is **17000** hours. \n\nThe code execution divided the original value 17059.87 by 1000 (yielding 17.05987), applied the `round()` function to obtain 17, then multiplied by 1000 to restore the scale, resulting in 17000. This aligns with standard rounding rules, as 17059.87 is closer to 17000 than 18000 when considering the nearest 1000-hour increment. ◥[code]◤',
 'GAIA',
 '17',
 'If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, h

In [40]:
res=new_res

In [14]:
json.loads(res[0][1])['answer']

'0'

In [15]:
import pickle
with open("/Users/chenyao/Documents/DeepLiterature/src/logs/smol_testres.pkl", "wb") as f:
    pickle.dump(res, f)

In [16]:
import json_repair

In [17]:
json_repair.loads("""{"answer": 0, "explanation": "The user's answer is incorrect. The standard answer is 2 because the stopping condition (consecutive iterations rounding to the same value) is met at \( n = 2 \), where \( x_2 \) and \( x_3 \) both round to -4.9361. The user incorrectly states \( n = 3 \), likely due to a miscalculation or misunderstanding of iteration indexing. The correct smallest \( n \) is 2, not 3."}""")

{'answer': 0,
 'explanation': "The user's answer is incorrect. The standard answer is 2 because the stopping condition (consecutive iterations rounding to the same value) is met at \\( n = 2 \\), where \\( x_2 \\) and \\( x_3 \\) both round to -4.9361. The user incorrectly states \\( n = 3 \\), likely due to a miscalculation or misunderstanding of iteration indexing. The correct smallest \\( n \\) is 2, not 3."}

In [30]:
res[0]

(None,
 '{"cot":"The user\'s answer of 17000 hours results from correctly rounding 17059.87 to the nearest 1000 hours. However, the question explicitly asks for the answer in \'thousand hours\' (i.e., the number of thousands), which requires dividing the total hours by 1000 after rounding. The standard answer is 17 (thousand hours), whereas the user provided 17000 (hours), missing the required unit conversion. Thus, the answer is incorrect.","answer":"0"}',
 '\n\n【Answer】The rounded result to the nearest 1000 hours (without commas) is **17000** hours. \n\nThe code execution divided the original value 17059.87 by 1000 (yielding 17.05987), applied the `round()` function to obtain 17, then multiplied by 1000 to restore the scale, resulting in 17000. This aligns with standard rounding rules, as 17059.87 is closer to 17000 than 18000 when considering the nearest 1000-hour increment. ◥[code]◤',
 'GAIA',
 'If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many

In [49]:
error_label=[]
def cal_accuracy(res):
    correct=0
    partial_correct=0
    all_num=0
    empty_num=0
    for i in range(len(res)):
        if res[i][2]!='':
            all_num+=1
            try:
                json_asnwer=json_repair.loads(res[i][1])['answer']
                # json_asnwer=json.loads(res[i][0])['answer']
                if json_asnwer==1 or json_asnwer=='1':
                    correct+=1
                elif json_asnwer==0.5:
                    partial_correct+=1
                else:
                    error_label.append(json_asnwer)
            except:
                
                print(res[i][1])
        else:
            empty_num+=1
    print("空值数",empty_num)
    return correct/all_num,partial_correct/all_num

In [50]:
math_res=[example for example in res if example[3]=='MATH']
gaia_re=[example for example in res if example[3]=='GAIA']
sp_res=[example for example in res if example[3]=='SimpleQA']
cal_accuracy(math_res),cal_accuracy(gaia_re),cal_accuracy(sp_res)

空值数 14
空值数 7
空值数 3


((0.8055555555555556, 0.0), (0.44, 0.0), (0.7659574468085106, 0.0))

In [51]:
cal_accuracy(res)

空值数 24


(0.7037037037037037, 0.0)

In [37]:
res[0]

(None,
 '{"cot":"The user\'s answer of 17000 hours results from correctly rounding 17059.87 to the nearest 1000 hours. However, the question explicitly asks for the answer in \'thousand hours\' (i.e., the number of thousands), which requires dividing the total hours by 1000 after rounding. The standard answer is 17 (thousand hours), whereas the user provided 17000 (hours), missing the required unit conversion. Thus, the answer is incorrect.","answer":"0"}',
 '\n\n【Answer】The rounded result to the nearest 1000 hours (without commas) is **17000** hours. \n\nThe code execution divided the original value 17059.87 by 1000 (yielding 17.05987), applied the `round()` function to obtain 17, then multiplied by 1000 to restore the scale, resulting in 17000. This aligns with standard rounding rules, as 17059.87 is closer to 17000 than 18000 when considering the nearest 1000-hour increment. ◥[code]◤',
 'GAIA',
 'If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many

In [60]:
for i in range(38,len(math_res)):
    if json_repair.loads(math_res[i][1])['answer']==0:
        print(math_res[i][5])
        print("="*50)
        print(math_res[i][1],"\n")
        print("="*50)
        print(math_res[i][2],"\n")
        print("="*50)
        print(math_res[i][4],"\n")
        print("="*50)
        print("idx:",i)
        break
        

In [24]:
json_repair.loads(res[0][1])['answer']

'0'

In [80]:
tmp_ls

[2]

In [81]:
tmp=0

In [88]:

for i in range(tmp,len(sp_res)):
    if json_repair.loads(sp_res[i][1])['answer']==0:
        print(sp_res[i][5])
        print("="*50)
        print(sp_res[i][1],"\n")
        print("="*50)
        print(sp_res[i][2],"\n")
        print("="*50)
        print(sp_res[i][4],"\n")
        print("="*50)
        tmp=i+1
        break

How many licenses to sell liquor did the council of Paris, Ontario, grant in 1850 when seven tavern keepers applied but were met with backlash from over 100 abolitionist villagers?
{"cot": "The user's response provides a detailed historical context and analysis of factors influencing the Paris, Ontario council's decision on liquor licenses in the 1850s, including legislative, sociopolitical, and archival considerations. However, it does not explicitly state a numerical answer to the question. The standard answer is '3', but the user's response omits any specific number, focusing instead on inferred rationale and evidentiary limitations. Since the question requires a number and the user failed to provide one, their answer is incorrect.", "answer": 0} 



【Answer】  
The reconciliation of primary source data with historical context to determine the Paris, Ontario council’s final decision on liquor licensing in the 1850s is hindered by the absence of digitized council minutes. However, the